# Objective 2 — Notebook 03: Spatiotemporal Alignment

**Project:** Evaluating Multi-Source Data Fusion and Uncertainty in Hyper-Local Flash-Flood Prediction for Hilly Regions of India (Uttarakhand)  
**Objective 2 Deliverable 3:** Geographically and temporally aligned base table adhering to a strictly causal timeline.

---

### Alignment Strategy:
1. **Primary Observation Grain**: One Station $\times$ One Hourly Timestamp.
2. **Spatial Join (Landslides)**: Computes Haversine distance from each station to all 5,444 landslide coordinates, aggregating density at 5 km and 10 km buffers.
3. **District Join**: Attaches static district-level hazard indicators (DFSI, flooded area %, historical casualties) via standardized district name.
4. **No Future Imputation**: Forward-filling limited to short sensor dropouts ($\le 3$ hours); strictly zero backward filling from future.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.config import CLEANED_DIR, ALIGNED_DIR
from src.spatial import compute_station_landslide_features
from src.temporal import generate_calendar_features

print("Alignment modules loaded.")

## 1. Load Cleaned Rainfall and Landslide Tables

In [ ]:
rain_df = pd.read_parquet(CLEANED_DIR / "rainfall_cleaned.parquet")
land_df = pd.read_parquet(CLEANED_DIR / "landslide_cleaned.parquet")
dist_df = pd.read_parquet(CLEANED_DIR / "district_supporting_cleaned.parquet")

print(f"Rainfall base records: {len(rain_df):,}")
print(f"Landslide points: {len(land_df):,}")

## 2. Compute Spatial Landslide Exposure per Station

In [ ]:
station_landslides = compute_station_landslide_features(rain_df, land_df)
print(f"Computed spatial features for {len(station_landslides)} stations:")
display(station_landslides.head(10))

## 3. Merge Spatial Features & District Metrics to Base Table

In [ ]:
aligned_base = rain_df.merge(
    station_landslides[["station", "dist_to_nearest_landslide_km", "landslide_count_5km", "landslide_count_10km", "is_landslide_hotspot"]],
    on="station",
    how="left"
)

aligned_base = aligned_base.merge(dist_df, on="district", how="left")
aligned_base = generate_calendar_features(aligned_base, datetime_col="datetime")

print(f"Aligned Base Table: {aligned_base.shape[0]:,} rows x {aligned_base.shape[1]} columns")
aligned_base.head()